In [1]:
import tensorflow as tf
from pathlib import Path
import numpy as np
import math
import h5py
import cv2

I0000 00:00:1793187710.739619    5533 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793187710.739979    5533 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1793187710.770277    5533 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793187711.614200    5533 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

In [2]:

root_dir = Path.cwd().parent

image_dir = root_dir / "data" / "TrainData" / "img"
mask_dir = root_dir / "data" / "TrainData" / "mask"

bad_ids = {
    2178, 2184, 225, 2250, 2251, 2252, 2253,
    2254, 2255, 2256, 2257, 2462, 2463, 2464
}

print("Image directory exists:", image_dir.exists())
print("Mask directory exists:", mask_dir.exists())


Image directory exists: True
Mask directory exists: True


# # Remove corrupted samples from both existing splits.

In [3]:
image_files = sorted(image_dir.glob("image_*.h5"))

sample_ids = []

for image_path in image_files:
    image_id = int(image_path.stem.replace("image_", ""))
    mask_path = mask_dir / f"mask_{image_id}.h5"

    if mask_path.exists():
        sample_ids.append(image_id)

print("Matched image-mask pairs:", len(sample_ids))

Matched image-mask pairs: 3799


In [4]:
from sklearn.model_selection import train_test_split

clean_ids = [
    image_id for image_id in sample_ids
    if image_id not in bad_ids
]

train_ids, val_ids = train_test_split(
    clean_ids,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print("Clean samples:", len(clean_ids))
print("Training samples:", len(train_ids))
print("Validation samples:", len(val_ids))
print("Overlap:", len(set(train_ids) & set(val_ids)))

Clean samples: 3785
Training samples: 3028
Validation samples: 757
Overlap: 0


### Validate the cleaned dataset

In [5]:

def check_sample(image_id):
    image_path = image_dir / f"image_{image_id}.h5"
    mask_path = mask_dir / f"mask_{image_id}.h5"

    if not image_path.exists() or not mask_path.exists():
        return False, "Missing image or mask"

    with h5py.File(image_path, "r") as f:
        image = f["img"][:]

    with h5py.File(mask_path, "r") as f:
        mask = f["mask"][:]

    if image.shape != (128, 128, 14):
        return False, f"Unexpected image shape: {image.shape}"

    if mask.shape != (128, 128):
        return False, f"Unexpected mask shape: {mask.shape}"

    if not np.isfinite(image).all():
        return False, "NaN or infinite image values"

    if np.any(np.abs(image) > np.finfo(np.float32).max):
        return False, "Values exceed float32 range"

    if not np.isin(mask, [0, 1]).all():
        return False, "Unexpected mask values"

    return True, "OK"


problems = []

for image_id in clean_ids:
    valid, message = check_sample(image_id)

    if not valid:
        problems.append((image_id, message))

print("Samples checked:", len(clean_ids))
print("Remaining problems:", len(problems))
print("Problem details:", problems[:10])


Samples checked: 3785
Remaining problems: 0
Problem details: []


In [6]:

def calculate_indices(image):
    """
    Input:  (128, 128, 14)
    Output: (128, 128, 3)
    """
    blue = image[:, :, 1]   # B2
    green = image[:, :, 2]  # B3
    red = image[:, :, 3]    # B4
    nir = image[:, :, 7]    # B8
    swir1 = image[:, :, 10] # B11
    swir2 = image[:, :, 11] # B12

    eps = 1e-6

    ndvi = (nir - red) / (nir + red + eps)
    ndmi = (nir - swir1) / (nir + swir1 + eps)
    nbr = (nir - swir2) / (nir + swir2 + eps)

    return np.stack([ndvi, ndmi, nbr], axis=-1)


In [7]:
sample_path = image_dir / "image_1.h5"

with h5py.File(sample_path, "r") as f:
    sample_image = f["img"][:].astype(np.float32)
with h5py.File(sample_path, "r") as f:
    sample_image = f["img"][:].astype(np.float32)

indices = calculate_indices(sample_image)

print("Original shape:", sample_image.shape)
print("Index shape:", indices.shape)
print("All values finite:", np.isfinite(indices).all())

for i, name in enumerate(["NDVI", "NDMI", "NBR"]):
    print(
        f"{name}: min={indices[:, :, i].min():.3f}, "
        f"max={indices[:, :, i].max():.3f}"
    )


Original shape: (128, 128, 14)
Index shape: (128, 128, 3)
All values finite: True
NDVI: min=-0.547, max=0.228
NDMI: min=-0.451, max=0.169
NBR: min=-0.667, max=0.231


In [8]:
def normalize_rgb(image):
    """Normalize B2, B3 and B4 to the range 0–1."""
    rgb = image[:, :, [1, 2, 3]].astype(np.float32)

    min_val = rgb.min(axis=(0, 1), keepdims=True)
    max_val = rgb.max(axis=(0, 1), keepdims=True)

    normalized = (rgb - min_val) / (max_val - min_val + 1e-6)

    return normalized

In [9]:
normalized_rgb = normalize_rgb(sample_image)

print("Normalized RGB shape:", normalized_rgb.shape)
print("Minimum:", normalized_rgb.min())
print("Maximum:", normalized_rgb.max())
print("All values finite:", np.isfinite(normalized_rgb).all())

Normalized RGB shape: (128, 128, 3)
Minimum: 0.0
Maximum: 0.99999976
All values finite: True


In [10]:
def calculate_grayscale(image):
    """Calculate grayscale from B2, B3 and B4."""
    blue = image[:, :, 1]
    green = image[:, :, 2]
    red = image[:, :, 3]

    grayscale = (blue + green + red) / 3.0

    return grayscale[:, :, np.newaxis]

In [11]:
grayscale = calculate_grayscale(sample_image)

print("Grayscale shape:", grayscale.shape)
print("Minimum:", grayscale.min())
print("Maximum:", grayscale.max())
print("All values finite:", np.isfinite(grayscale).all())

Grayscale shape: (128, 128, 1)
Minimum: 1.0214823
Maximum: 3.6857708
All values finite: True


In [12]:
from scipy.ndimage import gaussian_filter, median_filter

In [13]:
def calculate_filters(image):
    """Create Gaussian and median filtered grayscale features."""
    grayscale = calculate_grayscale(image)[:, :, 0]

    gaussian = gaussian_filter(grayscale, sigma=1)
    median = median_filter(grayscale, size=10)

    return np.stack([gaussian, median], axis=-1)

In [14]:
filtered = calculate_filters(sample_image)

print("Filtered shape:", filtered.shape)
print("All values finite:", np.isfinite(filtered).all())

for i, name in enumerate(["Gaussian", "Median"]):
    print(
        f"{name}: min={filtered[:, :, i].min():.3f}, "
        f"max={filtered[:, :, i].max():.3f}"
    )

Filtered shape: (128, 128, 2)
All values finite: True
Gaussian: min=1.056, max=3.348
Median: min=1.081, max=2.552


In [15]:
def calculate_gradients(image):
    """Calculate horizontal and vertical grayscale gradients."""
    grayscale = calculate_grayscale(image)[:, :, 0]

    vertical_gradient, horizontal_gradient = np.gradient(grayscale)

    return np.stack(
        [horizontal_gradient, vertical_gradient],
        axis=-1
    )

In [16]:
gradients = calculate_gradients(sample_image)

print("Gradient shape:", gradients.shape)
print("All values finite:", np.isfinite(gradients).all())

for i, name in enumerate(["Horizontal", "Vertical"]):
    print(
        f"{name}: min={gradients[:, :, i].min():.4f}, "
        f"max={gradients[:, :, i].max():.4f}"
    )

Gradient shape: (128, 128, 2)
All values finite: True
Horizontal: min=-1.0558, max=1.0136
Vertical: min=-0.9658, max=0.9393


In [17]:

def calculate_edges(image):
    """Detect edges in the grayscale image using Canny."""
    grayscale = calculate_grayscale(image)[:, :, 0]

    gray_min = grayscale.min()
    gray_max = grayscale.max()

    gray_scaled = (
        (grayscale - gray_min) / (gray_max - gray_min + 1e-6) * 255
    ).astype(np.uint8)

    edges = cv2.Canny(gray_scaled, 100, 200)

    return edges[:, :, np.newaxis].astype(np.float32) / 255.0

In [18]:
edges = calculate_edges(sample_image)

print("Edge shape:", edges.shape)
print("Unique values:", np.unique(edges))
print("All values finite:", np.isfinite(edges).all())
print("Edge pixels detected:", np.count_nonzero(edges))

Edge shape: (128, 128, 1)
Unique values: [0. 1.]
All values finite: True
Edge pixels detected: 982


In [19]:
def create_features(image):
    """Combine the 12 engineered features with the original 14 channels."""

    image = image.astype(np.float32)

    rgb = normalize_rgb(image)                         # 3 channels
    indices = calculate_indices(image)                 # 3 channels
    grayscale = calculate_grayscale(image)             # 1 channel
    filtered = calculate_filters(image)                # 2 channels
    gradients = calculate_gradients(image)              # 2 channels
    edges = calculate_edges(image)                      # 1 channel

    features = np.concatenate(
        [image, rgb, indices, grayscale, filtered, gradients, edges],
        axis=-1
    )

    return features

### Preparing the data

## Normalization

In [20]:
channel_sum = np.zeros(26, dtype=np.float64)
channel_sq_sum = np.zeros(26, dtype=np.float64)
pixel_count = 0

for i in range(len(train_ids)):
    image_id = train_ids[i]

    with h5py.File(image_dir / f"image_{image_id}.h5", "r") as f:
        image = f["img"][:]

    features = create_features(image).astype(np.float64)

    channel_sum += features.sum(axis=(0, 1))
    channel_sq_sum += (features ** 2).sum(axis=(0, 1))
    pixel_count += features.shape[0] * features.shape[1]

channel_mean = channel_sum / pixel_count
channel_var = channel_sq_sum / pixel_count - channel_mean ** 2
channel_std = np.sqrt(np.maximum(channel_var, 0))
channel_std = np.maximum(channel_std, 1e-6)

channel_mean = channel_mean.reshape(1, 1, 1, 26).astype(np.float32)
channel_std = channel_std.reshape(1, 1, 1, 26).astype(np.float32)

print("Normalization statistics ready.")
print("Mean shape:", channel_mean.shape)
print("Std shape:", channel_std.shape)

KeyboardInterrupt: 

## Callbacks

In [25]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "best_landslide_unet.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

# augmentation

### Update the generator

In [26]:

class LandslideDataGenerator(tf.keras.utils.Sequence):

    def __init__(
        self,
        sample_ids,
        batch_size=8,
        shuffle=False,
        mean=None,
        std=None,
        augment=False
    ):
        super().__init__()

        self.sample_ids = list(sample_ids)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.mean = mean
        self.std = std
        self.augment = augment

        self.on_epoch_end()

    def __len__(self):
        return math.ceil(len(self.sample_ids) / self.batch_size)

    def __getitem__(self, index):
        start = index * self.batch_size
        batch_ids = self.sample_ids[start:start + self.batch_size]

        images, masks = [], []

        for image_id in batch_ids:

            with h5py.File(
                image_dir / f"image_{image_id}.h5", "r"
            ) as f:
                image = f["img"][:]

            with h5py.File(
                mask_dir / f"mask_{image_id}.h5", "r"
            ) as f:
                mask = f["mask"][:]

            # Create the 26-channel feature image
            features = create_features(image)

            # Apply identical spatial transformations
            # to image features and segmentation mask
            if self.augment:

                if np.random.rand() < 0.5:
                    features = np.flip(features, axis=1)
                    mask = np.flip(mask, axis=1)

                if np.random.rand() < 0.5:
                    features = np.flip(features, axis=0)
                    mask = np.flip(mask, axis=0)

                k = np.random.randint(0, 4)

                if k:
                    features = np.rot90(features, k=k, axes=(0, 1))
                    mask = np.rot90(mask, k=k, axes=(0, 1))

            images.append(features)
            masks.append(mask[..., np.newaxis].astype(np.float32))

        X = np.stack(images).astype(np.float32)
        y = np.stack(masks).astype(np.float32)

        # Use the training-set normalization statistics
        if self.mean is not None and self.std is not None:
            X = (X - self.mean) / self.std

        return X, {
            "coarse_mask": y,
            "medium_mask": y,
            "final_mask": y
        }

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.sample_ids)


### Recreate the generators

In [27]:
train_data = LandslideDataGenerator(
    train_ids,
    batch_size=8,
    shuffle=True,
    mean=channel_mean,
    std=channel_std,
    augment=True
)

val_data = LandslideDataGenerator(
    val_ids,
    batch_size=8,
    shuffle=False,
    mean=channel_mean,
    std=channel_std,
    augment=False
)

print("Training batches:", len(train_data))
print("Validation batches:", len(val_data))
print("Training augmentation:", train_data.augment)
print("Validation augmentation:", val_data.augment)

Training batches: 379
Validation batches: 95
Training augmentation: True
Validation augmentation: False


In [28]:
def focal_dice_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)

    # Binary focal loss
    bce = tf.keras.backend.binary_crossentropy(y_true, y_pred)

    p_t = y_true * y_pred + (1.0 - y_true) * (1.0 - y_pred)
    alpha_t = y_true * 0.25 + (1.0 - y_true) * 0.75

    focal = alpha_t * tf.pow(1.0 - p_t, 2.0) * bce
    focal = tf.reduce_mean(focal)

    # Dice loss
    y_true_flat = tf.reshape(y_true, [-1])
    y_pred_flat = tf.reshape(y_pred, [-1])

    intersection = tf.reduce_sum(y_true_flat * y_pred_flat)

    dice = (
        2.0 * intersection + 1e-6
    ) / (
        tf.reduce_sum(y_true_flat) +
        tf.reduce_sum(y_pred_flat) + 1e-6
    )

    return focal + (1.0 - dice)

In [ ]:
from tensorflow.keras import layers, Model

# Input: 128 x 128 x 26 engineered features
INPUT_SHAPE = (128, 128, 26)


def residual_block(x, filters):
    """Residual block with parallel 3x3 and 5x5 convolutions."""

    shortcut = layers.Conv2D(
        filters, 1, padding="same", use_bias=False
    )(x)
    shortcut = layers.BatchNormalization()(shortcut)

    conv3 = layers.Conv2D(
        filters, 3, padding="same", use_bias=False
    )(x)
    conv3 = layers.BatchNormalization()(conv3)
    conv3 = layers.Activation("relu")(conv3)

    conv5 = layers.Conv2D(
        filters, 5, padding="same", use_bias=False
    )(x)
    conv5 = layers.BatchNormalization()(conv5)
    conv5 = layers.Activation("relu")(conv5)

    x = layers.Add()([conv3, conv5])
    x = layers.Add()([x, shortcut])
    x = layers.Activation("relu")(x)

    return x


def attention_block(x, num_heads=4):
    """Multi-head self-attention over spatial positions."""

    shape = x.shape
    height, width, channels = shape[1], shape[2], shape[3]

    tokens = layers.Reshape((height * width, channels))(x)

    attention = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=int(channels) // num_heads
    )(tokens, tokens)

    tokens = layers.Add()([tokens, attention])
    tokens = layers.LayerNormalization()(tokens)

    x = layers.Reshape((height, width, channels))(tokens)

    return x


def decoder_block(x, skip, filters):
    x = layers.UpSampling2D(size=(2, 2))(x)
    x = layers.Concatenate()([x, skip])
    x = residual_block(x, filters)
    return x


def segmentation_head(x, target_size, name):
    """Create a segmentation prediction at 128 x 128."""

    x = layers.Conv2D(1, 1, activation="sigmoid")(x)

    if target_size != (128, 128):
        x = layers.Resizing(128, 128)(x)

    return layers.Activation("linear", name=name)(x)


def build_improved_unet(input_shape=INPUT_SHAPE):
    inputs = layers.Input(shape=input_shape)

    # Encoder
    e1 = residual_block(inputs, 32)       # 128 x 128
    p1 = layers.MaxPooling2D()(e1)

    e2 = residual_block(p1, 64)           # 64 x 64
    p2 = layers.MaxPooling2D()(e2)

    e3 = residual_block(p2, 128)          # 32 x 32
    p3 = layers.MaxPooling2D()(e3)

    e4 = residual_block(p3, 256)          # 16 x 16
    p4 = layers.MaxPooling2D()(e4)

    # Bottleneck + attention
    b = residual_block(p4, 512)           # 8 x 8
    b = attention_block(b, num_heads=4)

    # Decoder
    d4 = decoder_block(b, e4, 256)        # 16 x 16
    d3 = decoder_block(d4, e3, 128)       # 32 x 32
    d2 = decoder_block(d3, e2, 64)        # 64 x 64
    d1 = decoder_block(d2, e1, 32)        # 128 x 128

    # Three supervised segmentation outputs
    output_coarse = segmentation_head(
        d3, (32, 32), "coarse_mask"
    )
    output_medium = segmentation_head(
        d2, (64, 64), "medium_mask"
    )
    output_final = layers.Conv2D(
        1, 1, activation="sigmoid", name="final_mask"
    )(d1)

    return Model(
        inputs=inputs,
        outputs=[output_coarse, output_medium, output_final],
        name="Improved_Landslide_UNet"
    )


model = build_improved_unet()
model.summary()

Model: "Improved_Landslide_UNet"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 26)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 128, 128,  │      7,488 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 128, 128,  │     20,800 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        128 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        128 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 128, 128,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 128, 128,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 128, 128,  │        832 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 128, 128,  │          0 │ activation[0][0], │
│                     │ 32)               │            │ activation_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 128, 128,  │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 128, 128,  │          0 │ add[0][0],        │
│                     │ 32)               │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 128, 128,  │          0 │ add_1[0][0]       │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 64, 64,    │          0 │ activation_2[0][… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 64, 64,    │     18,432 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 64, 64,    │     51,200 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        256 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        256 │ conv2d_5[0][0]  

 Total params: 16,330,659 (62.30 MB)

 Trainable params: 16,321,827 (62.26 MB)

 Non-trainable params: 8,832 (34.50 KB)

In [30]:
def focal_iou_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)

    # Binary focal loss
    bce = tf.keras.backend.binary_crossentropy(y_true, y_pred)

    p_t = y_true * y_pred + (1.0 - y_true) * (1.0 - y_pred)
    alpha_t = y_true * 0.25 + (1.0 - y_true) * 0.75

    focal = alpha_t * tf.square(1.0 - p_t) * bce
    focal = tf.reduce_mean(focal)

    # IoU loss
    intersection = tf.reduce_sum(y_true * y_pred)
    union = (
        tf.reduce_sum(y_true)
        + tf.reduce_sum(y_pred)
        - intersection
    )

    iou = (intersection + 1e-6) / (union + 1e-6)

    return focal + (1.0 - iou)

In [31]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss={
        "coarse_mask": focal_iou_loss,
        "medium_mask": focal_iou_loss,
        "final_mask": focal_iou_loss,
    },
    loss_weights={
        "coarse_mask": 0.2,
        "medium_mask": 0.3,
        "final_mask": 1.0,
    },
    metrics={
        "final_mask": [
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
        ]
    }
)

print("Improved U-Net compiled successfully.")

Improved U-Net compiled successfully.


In [32]:
train_data = LandslideDataGenerator(
    train_ids, batch_size=8, shuffle=True,
    mean=channel_mean, std=channel_std, augment=True
)

val_data = LandslideDataGenerator(
    val_ids, batch_size=8, shuffle=False,
    mean=channel_mean, std=channel_std, augment=False
)

In [33]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=30,
    callbacks=callbacks
)

Epoch 1/30


I0000 00:00:1793153436.250690    6055 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
E0000 00:00:1793153444.875516    6055 util.cc:131] oneDNN supports DT_INT32 only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


379/379 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - coarse_mask_loss: 0.8379 - final_mask_loss: 0.8352 - final_mask_precision: 0.3486 - final_mask_recall: 0.6766 - loss: 1.2417 - medium_mask_loss: 0.7979
Epoch 1: val_loss improved from None to 1.05733, saving model to best_landslide_unet.keras

Epoch 1: finished saving model to best_landslide_unet.keras
379/379 ━━━━━━━━━━━━━━━━━━━━ 775s 2s/step - coarse_mask_loss: 0.8379 - final_mask_loss: 0.8352 - final_mask_precision: 0.3486 - final_mask_recall: 0.6766 - loss: 1.2417 - medium_mask_loss: 0.7979 - val_coarse_mask_loss: 0.7142 - val_final_mask_loss: 0.7087 - val_final_mask_precision: 0.7401 - val_final_mask_recall: 0.5693 - val_loss: 1.0573 - val_medium_mask_loss: 0.6827 - learning_rate: 1.0000e-04
Epoch 2/30
379/379 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - coarse_mask_loss: 0.6708 - final_mask_loss: 0.6515 - final_mask_precision: 0.6215 - final_mask_recall: 0.6505 - loss: 0.9775 - medium_mask_loss: 0.6388
Epoch 2: val_loss improved from 1.05733 to 0.88

In [34]:
np.savez(
    "normalization_stats_26ch.npz",
    mean=channel_mean,
    std=channel_std
)

print("Saved 26-channel normalization statistics.")

Saved 26-channel normalization statistics.
